<a href="https://colab.research.google.com/github/rjlobosco/unifesp/blob/main/Aula_Cholesky_Calculo_Numerico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cálculo Numérico: Decomposição de Cholesky

**Objetivos:** Identificar quando o método de Cholesky pode ser aplicado; fatorar uma matriz $A=LL^T$ e resolver $Ax=b$;


## 0. Preparação e diagnóstico

Uma matriz simétrica definida positiva (SDP) admite a aplicação do método de Cholesky. Já o método de Jacobi é um método iterativo que produz aproximações sucessivas e sua convergência depende da matriz. Vamos usar os dois métodos no mesmo sistema e avaliar as respostas por meio do resíduo.

In [1]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

### Revisão rápida

Para um sistema linear $Ax=b$, uma solução exata satisfaz a equação. Para uma aproximação $\hat{x}$, o **resíduo** é $r=b-A\hat{x}$. Um resíduo pequeno indica que a aproximação satisfaz bem o sistema (considerando a escala dos dados), mas não é sempre sinônimo de erro relativo pequeno se $A$ for mal condicionada.

**Perguntas iniciais:** O que significa $A=A^T$? Como verificar se uma matriz é definida positiva? Qual é a diferença entre um método direto e um iterativo?

In [2]:
A = np.array([[4., 1., 1.],
              [1., 3., 0.],
              [1., 0., 2.]])
b = np.array([6., 5., 5.])
print("A =\n", A)
print("b =", b)
print("Simétrica?", np.allclose(A, A.T))
print("Autovalores de A:", np.linalg.eigvalsh(A))
x_ref = np.linalg.solve(A, b)
print("Solução de referência:", x_ref)

A =
 [[4. 1. 1.]
 [1. 3. 0.]
 [1. 0. 2.]]
b = [6. 5. 5.]
Simétrica? True
Autovalores de A: [1.467911 2.652704 4.879385]
Solução de referência: [0.578947 1.473684 2.210526]


### A- Como verificar a simetria e calcular os autovalores

Uma matriz quadrada $A$ é **simétrica** quando A é igual à sua transposta:

$$
A = A^T
$$

A transposta é obtida trocando as linhas pelas colunas. Assim, $A$ é simétrica quando $a_{ij}=a_{ji}$ para todos os índices $i$ e $j$.

Os **autovalores** de $A$ são os valores $\lambda$ para os quais existe um vetor não nulo $v$ que satisfaz:

$$
Av = \lambda v
$$

Em geral, eles são calculados resolvendo a equação característica:

$$
\det(A-\lambda I)=0
$$

em que $I$ é a matriz identidade. Para uma matriz real e simétrica, podemos calcular os autovalores em Python usando `numpy.linalg.eigvalsh(A)`.



### B -Como verificar se uma matriz é definida positiva: menores principais

Para verificar se uma matriz simétrica é definida positiva, podemos usar o **critério de Sylvester**. Ele afirma que a matriz é definida positiva se todos os seus menores principais líderes forem positivos:

$$
\Delta_1>0,\quad \Delta_2>0,\quad \ldots,\quad \Delta_n>0.
$$

Os menores principais líderes são os determinantes dos blocos quadrados que começam no canto superior esquerdo da matriz.

Considere, por exemplo,

$$
A=
\begin{bmatrix}
4 & 1 & 1\\
1 & 3 & 0\\
1 & 0 & 2
\end{bmatrix}.
$$

O primeiro menor principal líder é o elemento da posição $(1,1)$:

$$
\Delta_1=4>0.
$$

O segundo é o determinante do bloco superior esquerdo de ordem 2:

$$
\Delta_2=
\begin{vmatrix}
4 & 1\\
1 & 3
\end{vmatrix}
=4\cdot3-1\cdot1
=11>0.
$$

O terceiro é o determinante da matriz completa:

$$
\begin{aligned}
\Delta_3=\det(A)
&=
4\begin{vmatrix}
3 & 0\\
0 & 2
\end{vmatrix}
-
1\begin{vmatrix}
1 & 0\\
1 & 2
\end{vmatrix}
+
1\begin{vmatrix}
1 & 3\\
1 & 0
\end{vmatrix}\\
&=4(3\cdot2-0\cdot0)
-(1\cdot2-0\cdot1)
+(1\cdot0-3\cdot1)\\
&=24-2-3\\
&=19>0.
\end{aligned}
$$

Como $A$ é simétrica e seus menores principais líderes são positivos, o critério de Sylvester garante que $A$ é definida positiva. Portanto, ela admite decomposição de Cholesky:

$$
A=LL^T.
$$


## 1. Decomposição de Cholesky

A decomposição de Cholesky é um método para fatorar uma matriz e, em seguida, resolver sistemas lineares. Ela pode ser aplicada quando a matriz $A$ é **quadrada, real, simétrica e definida positiva**.

Nessas condições, existe uma única matriz triangular inferior $L$, com elementos positivos na diagonal principal, tal que

$$
A = LL^T.
$$

Aqui, $L^T$ representa a transposta de $L$. Como $L$ é triangular inferior, todos os elementos acima de sua diagonal principal são iguais a zero. Por exemplo,

$$
L =
\begin{bmatrix}
l_{11} & 0 & 0 \\
l_{21} & l_{22} & 0 \\
l_{31} & l_{32} & l_{33}
\end{bmatrix}.
$$

A condição de diagonal positiva significa que

$$
l_{11} > 0,\qquad l_{22} > 0,\qquad l_{33} > 0.
$$

### Como a fatoração ajuda a resolver um sistema linear?

Considere o sistema

$$
Ax = b,
$$

em que $A$ é a matriz dos coeficientes, $x$ é o vetor de incógnitas e $b$ é o vetor de termos independentes.

Como $A=LL^T$, podemos substituir $A$ na equação:

$$
LL^T x = b.
$$

Para resolver esse sistema, introduzimos um vetor intermediário $y$ e resolvemos dois sistemas triangulares, um após o outro:

1. **Substituição progressiva:**

   $$Ly=b.$$

   Como $L$ é triangular inferior, calculamos as componentes de $y$ começando pela primeira linha e seguindo de cima para baixo.

2. **Substituição regressiva:**

   $$L^T x=y.$$

   Como $L^T$ é triangular superior, calculamos as componentes de $x$ começando pela última linha e seguindo de baixo para cima.

Assim, em vez de resolver diretamente o sistema original, resolvemos duas etapas mais simples:

$$
\boxed{Ly=b}
\qquad \longrightarrow \qquad
\boxed{L^T x=y}.
$$

### Por que usar esse método?

Depois de calcular a fatoração $A=LL^T$, podemos reutilizá-la para resolver sistemas com a mesma matriz $A$ e diferentes vetores $b$. Isso evita refazer toda a fatoração a cada novo sistema.

A decomposição também aproveita a simetria da matriz e não requer o cálculo da matriz inversa. Em geral, para resolver $Ax=b$, é mais eficiente e numericamente adequado usar a fatoração e resolver os sistemas triangulares do que calcular $A^{-1}$.

**Resumo:** Cholesky transforma uma matriz simétrica definida positiva em um produto de duas matrizes triangulares. Com isso, resolver $Ax=b$ torna-se resolver sucessivamente $Ly=b$ e $L^T x=y$.

### Resolução do sistema usando a fatoração de Cholesky

Considere o sistema linear

$$
Ax=b.
$$

Se a matriz $A$ admite a decomposição de Cholesky, podemos escrevê-la como

$$
A=LL^T,
$$

em que $L$ é uma matriz triangular inferior e $L^T$ é sua transposta, uma matriz triangular superior.

Substituindo $A=LL^T$ no sistema original, obtemos

$$
LL^T x=b.
$$

Para resolver essa equação em duas etapas, introduzimos um vetor intermediário $y$:

$$
y=L^T x.
$$

Assim, o sistema fica dividido em:

$$
Ly=b
$$

e, depois,

$$
L^T x=y.
$$

### 1. Substituição progressiva: resolver $Ly=b$

Como $L$ é triangular inferior, os elementos acima da diagonal principal são nulos. Por isso, podemos calcular as componentes de $y$ começando pela primeira equação e seguindo de cima para baixo.

Por exemplo, para

$$
L=
\begin{bmatrix}
l_{11} & 0 & 0\\
l_{21} & l_{22} & 0\\
l_{31} & l_{32} & l_{33}
\end{bmatrix},
\qquad
y=
\begin{bmatrix}
y_1\\
y_2\\
y_3
\end{bmatrix},
\qquad
b=
\begin{bmatrix}
b_1\\
b_2\\
b_3
\end{bmatrix},
$$

o sistema $Ly=b$ corresponde a

$$
\begin{aligned}
l_{11}y_1 &= b_1,\\
l_{21}y_1+l_{22}y_2 &= b_2,\\
l_{31}y_1+l_{32}y_2+l_{33}y_3 &= b_3.
\end{aligned}
$$

Resolvemos as equações nesta ordem:

$$
y_1=\frac{b_1}{l_{11}},
$$

$$
y_2=\frac{b_2-l_{21}y_1}{l_{22}},
$$

$$
y_3=\frac{b_3-l_{31}y_1-l_{32}y_2}{l_{33}}.
$$

Cada nova componente usa os valores já calculados nas equações anteriores. Esse processo é chamado de **substituição progressiva**.

### 2. Substituição regressiva: resolver $L^T x=y$

Depois de encontrar $y$, resolvemos

$$
L^T x=y.
$$

A matriz $L^T$ é triangular superior. Portanto, começamos pela última equação e seguimos de baixo para cima.

Usando a matriz $L$ anterior,

$$
L^T=
\begin{bmatrix}
l_{11} & l_{21} & l_{31}\\
0 & l_{22} & l_{32}\\
0 & 0 & l_{33}
\end{bmatrix}.
$$

O sistema $L^T x=y$ é

$$
\begin{aligned}
l_{11}x_1+l_{21}x_2+l_{31}x_3 &= y_1,\\
l_{22}x_2+l_{32}x_3 &= y_2,\\
l_{33}x_3 &= y_3.
\end{aligned}
$$

Começamos pela última equação:

$$
x_3=\frac{y_3}{l_{33}}.
$$

Em seguida, usamos $x_3$ para calcular $x_2$:

$$
x_2=\frac{y_2-l_{32}x_3}{l_{22}}.
$$

Por fim, calculamos $x_1$:

$$
x_1=\frac{y_1-l_{21}x_2-l_{31}x_3}{l_{11}}.
$$

Esse processo é chamado de **substituição regressiva**.

### Por que usar esse método?

Depois de fatorar $A$ como $LL^T$, resolvemos dois sistemas triangulares simples. Se precisarmos resolver vários sistemas com a mesma matriz $A$ e diferentes vetores $b$, podemos reutilizar a fatoração e realizar somente as substituições.

Esse procedimento aproveita a simetria de $A$ e evita calcular a matriz inversa. Para resolver $Ax=b$, usamos a fatoração e as substituições triangulares, em vez de calcular $A^{-1}b$.

**Resumo:** a decomposição de Cholesky transforma o sistema $Ax=b$ em duas etapas:

$$
\boxed{Ly=b}
\qquad\longrightarrow\qquad
\boxed{L^T x=y}.
$$

A primeira etapa é resolvida de cima para baixo; a segunda, de baixo para cima.

### Exemplo numérico: fatoração de Cholesky e resolução do sistema

Considere o sistema linear

$$
Ax=b,
$$

com

$$
A=
\begin{bmatrix}
4 & 1\\
1 & 3
\end{bmatrix},
\qquad
b=
\begin{bmatrix}
6\\
5
\end{bmatrix}.
$$

Nosso objetivo é encontrar o vetor

$$
x=
\begin{bmatrix}
x_1\\
x_2
\end{bmatrix}
$$

que satisfaz $Ax=b$.

#### 1. Verificar se Cholesky pode ser aplicada

A matriz $A$ é simétrica porque os elementos fora da diagonal principal são iguais:

$$
a_{12}=a_{21}=1.
$$

Além disso, seus menores principais líderes são positivos:

$$
\Delta_1=4>0,
$$

e

$$
\Delta_2=\det(A)
=
\begin{vmatrix}
4 & 1\\
1 & 3
\end{vmatrix}
=4\cdot3-1\cdot1
=11>0.
$$

Pelo critério de Sylvester, $A$ é definida positiva. Portanto, podemos calcular sua decomposição de Cholesky.

#### 2. Calcular a fatoração $A=LL^T$

Procuramos uma matriz triangular inferior da forma

$$
L=
\begin{bmatrix}
l_{11} & 0\\
l_{21} & l_{22}
\end{bmatrix},
$$

tal que

$$
LL^T=A.
$$

A transposta de $L$ é

$$
L^T=
\begin{bmatrix}
l_{11} & l_{21}\\
0 & l_{22}
\end{bmatrix}.
$$

Multiplicando as matrizes:

Utilizamos, Elemento da primeira linha e primeira coluna, Elemento da primeira linha e segunda coluna, Elemento da segunda linha e primeira coluna, Elemento da segunda linha e segunda coluna.

$$
LL^T
=
\begin{bmatrix}
l_{11}^2 & l_{11}l_{21}\\
l_{11}l_{21} & l_{21}^2+l_{22}^2
\end{bmatrix}.
$$

Igualamos os elementos de $LL^T$ aos elementos correspondentes de $A$:

$$
\begin{aligned}
l_{11}^2 &= 4,\\
l_{11}l_{21} &= 1,\\
l_{21}^2+l_{22}^2 &= 3.
\end{aligned}
$$

Como a diagonal de $L$ deve ser positiva, escolhemos a raiz positiva:

$$
l_{11}=\sqrt{4}=2.
$$

Em seguida, calculamos $l_{21}$:

$$
2l_{21}=1
\quad\Longrightarrow\quad
l_{21}=\frac{1}{2}.
$$

Por fim, calculamos $l_{22}$:

$$
\left(\frac{1}{2}\right)^2+l_{22}^2=3,
$$

$$
\frac{1}{4}+l_{22}^2=3
\quad\Longrightarrow\quad
l_{22}^2=\frac{11}{4}.
$$

Como $l_{22}>0$,

$$
l_{22}=\frac{\sqrt{11}}{2}.
$$

Portanto,

$$
L=
\begin{bmatrix}
2 & 0\\
\frac{1}{2} & \frac{\sqrt{11}}{2}
\end{bmatrix}.
$$

Podemos conferir a fatoração multiplicando $L$ por $L^T$:

$$
LL^T
=
\begin{bmatrix}
2 & 0\\
\frac{1}{2} & \frac{\sqrt{11}}{2}
\end{bmatrix}
\begin{bmatrix}
2 & \frac{1}{2}\\
0 & \frac{\sqrt{11}}{2}
\end{bmatrix}
=
\begin{bmatrix}
4 & 1\\
1 & \frac{1}{4}+\frac{11}{4}
\end{bmatrix}
=
\begin{bmatrix}
4 & 1\\
1 & 3
\end{bmatrix}
=A.
$$

#### 3. Primeira etapa: resolver $Ly=b$

Como $A=LL^T$, o sistema $Ax=b$ pode ser escrito como

$$
LL^T x=b.
$$

Introduzimos o vetor intermediário $y=L^T x$ e resolvemos primeiro

$$
Ly=b.
$$

Substituindo os valores de $L$ e $b$:

$$
\begin{bmatrix}
2 & 0\\
\frac{1}{2} & \frac{\sqrt{11}}{2}
\end{bmatrix}
\begin{bmatrix}
y_1\\
y_2
\end{bmatrix}
=
\begin{bmatrix}
6\\
5
\end{bmatrix}.
$$

Esse produto matricial corresponde ao sistema

$$
\begin{aligned}
2y_1 &= 6,\\
\frac{1}{2}y_1+\frac{\sqrt{11}}{2}y_2 &= 5.
\end{aligned}
$$

Começamos pela primeira equação:

$$
y_1=\frac{6}{2}=3.
$$

Substituímos $y_1=3$ na segunda equação:

$$
\frac{1}{2}(3)+\frac{\sqrt{11}}{2}y_2=5.
$$

Isolando $y_2$:

$$
\frac{\sqrt{11}}{2}y_2
=5-\frac{3}{2}
=\frac{7}{2},
$$

$$
y_2=\frac{7}{\sqrt{11}}.
$$

Assim, o vetor intermediário é

$$
y=
\begin{bmatrix}
3\\
\frac{7}{\sqrt{11}}
\end{bmatrix}.
$$

Essa etapa é chamada de **substituição progressiva**, pois calculamos primeiro $y_1$ e depois usamos esse resultado para calcular $y_2$.

#### 4. Segunda etapa: resolver $L^T x=y$

Agora resolvemos

$$
L^T x=y.
$$

Substituindo os valores:

$$
\begin{bmatrix}
2 & \frac{1}{2}\\
0 & \frac{\sqrt{11}}{2}
\end{bmatrix}
\begin{bmatrix}
x_1\\
x_2
\end{bmatrix}
=
\begin{bmatrix}
3\\
\frac{7}{\sqrt{11}}
\end{bmatrix}.
$$

O sistema correspondente é

$$
\begin{aligned}
2x_1+\frac{1}{2}x_2 &= 3,\\
\frac{\sqrt{11}}{2}x_2 &= \frac{7}{\sqrt{11}}.
\end{aligned}
$$

Começamos pela última equação:

$$
x_2
=
\frac{7}{\sqrt{11}}\cdot\frac{2}{\sqrt{11}}
=
\frac{14}{11}.
$$

Substituímos esse valor na primeira equação:

$$
2x_1+\frac{1}{2}\left(\frac{14}{11}\right)=3.
$$

Como

$$
\frac{1}{2}\left(\frac{14}{11}\right)=\frac{7}{11}
\quad\text{e}\quad
3=\frac{33}{11},
$$

temos

$$
2x_1=\frac{33}{11}-\frac{7}{11}
=\frac{26}{11},
$$

$$
x_1=\frac{13}{11}.
$$

Essa etapa é chamada de **substituição regressiva**, pois começamos pela última equação e seguimos de baixo para cima.

#### 5. Solução e verificação

A solução encontrada é

$$
x=
\begin{bmatrix}
\frac{13}{11}\\
\frac{14}{11}
\end{bmatrix}.
$$

Para conferir, substituímos o vetor na equação original:

$$
Ax
=
\begin{bmatrix}
4 & 1\\
1 & 3
\end{bmatrix}
\begin{bmatrix}
\frac{13}{11}\\
\frac{14}{11}
\end{bmatrix}
=
\begin{bmatrix}
\frac{52}{11}+\frac{14}{11}\\
\frac{13}{11}+\frac{42}{11}
\end{bmatrix}
=
\begin{bmatrix}
6\\
5
\end{bmatrix}
=b.
$$

Portanto, o vetor encontrado satisfaz o sistema.

**Resumo do procedimento:**

$$
A=LL^T,
\qquad
Ly=b,
\qquad
L^T x=y.
$$

Primeiro fatoramos $A$. Depois, resolvemos $Ly=b$ por substituição progressiva e, por fim, resolvemos $L^T x=y$ por substituição regressiva.

### 1.1 Derivação elemento a elemento

Queremos encontrar uma matriz triangular inferior $L$ tal que

$$
A=LL^T.
$$

Como $L$ é triangular inferior, seus elementos acima da diagonal principal são nulos. Por exemplo, para uma matriz de ordem $3$,

$$
L=
\begin{bmatrix}
l_{11} & 0 & 0\\
l_{21} & l_{22} & 0\\
l_{31} & l_{32} & l_{33}
\end{bmatrix}.
$$

O elemento da posição $(i,j)$ do produto $LL^T$ é calculado multiplicando a linha $i$ de $L$ pela linha $j$ de $L$. Assim,

$$
a_{ij}
=
\sum_{k=1}^{\min(i,j)} l_{ik}l_{jk}.
$$

Essa relação permite calcular os elementos de $L$ usando os elementos conhecidos de $A$. O cálculo é feito coluna por coluna, começando pela diagonal.

#### Elementos da diagonal de $L$

Quando $i=j$, temos

$$
a_{jj}
=
\sum_{k=1}^{j} l_{jk}^2.
$$

Separando o último termo da soma,

$$
a_{jj}
=
\sum_{k=1}^{j-1} l_{jk}^2+l_{jj}^2.
$$

Isolando $l_{jj}$:

$$
l_{jj}
=
\sqrt{
a_{jj}
-
\sum_{k=1}^{j-1} l_{jk}^2
}.
$$

Escolhemos a raiz positiva para que a diagonal de $L$ seja positiva.

#### Elementos abaixo da diagonal de $L$

Para $i>j$, a relação entre os elementos de $A$ e de $L$ é

$$
a_{ij}
=
\sum_{k=1}^{j} l_{ik}l_{jk}.
$$

Separando o termo correspondente a $k=j$,

$$
a_{ij}
=
\sum_{k=1}^{j-1} l_{ik}l_{jk}
+
l_{ij}l_{jj}.
$$

Como $l_{jj}>0$, podemos isolar $l_{ij}$:

$$
l_{ij}
=
\frac{
a_{ij}
-
\displaystyle\sum_{k=1}^{j-1}l_{ik}l_{jk}
}{
l_{jj}
},
\qquad i>j.
$$

#### Ordem dos cálculos

Para cada coluna $j$:

1. Calculamos o elemento diagonal $l_{jj}$.
2. Usamos esse valor para calcular os elementos abaixo dele, isto é, $l_{ij}$ para $i>j$.
3. Avançamos para a próxima coluna.

Cada elemento é calculado a partir de valores que já foram determinados anteriormente.

#### Condição para a raiz

Em cada cálculo da diagonal, o valor dentro da raiz deve ser positivo:

$$
a_{jj}
-
\sum_{k=1}^{j-1}l_{jk}^2>0.
$$

Se esse valor for zero ou negativo, a fatoração de Cholesky padrão não pode prosseguir nos números reais. Isso pode indicar que a matriz não é definida positiva ou que há problemas de arredondamento ou de dados.

**Em resumo:** os elementos da diagonal de $L$ são calculados com uma raiz quadrada; os elementos abaixo da diagonal são calculados por divisão pelo elemento diagonal correspondente. Os elementos acima da diagonal são iguais a zero.

In [3]:
def cholesky_manual(A):
    """Fatora A = L L.T para A simétrica definida positiva."""
    A = np.asarray(A, dtype=float)
    n, m = A.shape
    if n != m:
        raise ValueError("A matriz deve ser quadrada.")
    if not np.allclose(A, A.T):
        raise ValueError("A matriz deve ser simétrica.")
    L = np.zeros_like(A)
    for i in range(n):
        for j in range(i + 1):
            s = sum(L[i, k] * L[j, k] for k in range(j))
            if i == j:
                d = A[i, i] - s
                if d <= 0:
                    raise ValueError(f"Pivô não positivo na posição {i}: {d}")
                L[i, j] = np.sqrt(d)
            else:
                L[i, j] = (A[i, j] - s) / L[j, j]
    return L

In [4]:
L = cholesky_manual(A)
print("L =\n", L)
print("L @ L.T =\n", L @ L.T)
print("Erro de reconstrução ||A - LLᵀ||∞ =", np.linalg.norm(A - L @ L.T, ord=np.inf))

L =
 [[ 2.        0.        0.      ]
 [ 0.5       1.658312  0.      ]
 [ 0.5      -0.150756  1.314257]]
L @ L.T =
 [[ 4.  1.  1.]
 [ 1.  3. -0.]
 [ 1. -0.  2.]]
Erro de reconstrução ||A - LLᵀ||∞ = 2.4691655676476636e-18


### 1.2 Resolver os dois sistemas triangulares

Implementaremos substituição progressiva e regressiva. Em ambos, cada incógnita é calculada usando as anteriores já encontradas.

In [5]:
def forward_substitution(L, b):
    L, b = np.asarray(L, float), np.asarray(b, float)
    n = len(b)
    y = np.zeros(n)
    for i in range(n):
        y[i] = (b[i] - L[i, :i] @ y[:i]) / L[i, i]
    return y

def backward_substitution(U, y):
    U, y = np.asarray(U, float), np.asarray(y, float)
    n = len(y)
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x

y = forward_substitution(L, b)
x_chol = backward_substitution(L.T, y)
print("y =", y)
print("x por Cholesky =", x_chol)
print("Resíduo b - Ax =", b - A @ x_chol)
print("Confere com NumPy?", np.allclose(x_chol, x_ref))

y = [3.       2.110579 2.905201]
x por Cholesky = [0.578947 1.473684 2.210526]
Resíduo b - Ax = [0. 0. 0.]
Confere com NumPy? True


### 1.3 Conferência com NumPy e custo

`numpy.linalg.cholesky` retorna $L$ triangular inferior. Para resolver, use duas chamadas a `numpy.linalg.solve`; evite formar a inversa de $A$.

In [6]:
L_np = np.linalg.cholesky(A)
y_np = np.linalg.solve(L_np, b)
x_np = np.linalg.solve(L_np.T, y_np)
print("L do NumPy =\n", L_np)
print("Solução =", x_np)

L do NumPy =
 [[ 2.        0.        0.      ]
 [ 0.5       1.658312  0.      ]
 [ 0.5      -0.150756  1.314257]]
Solução = [0.578947 1.473684 2.210526]


### Exercício final: decomposição de Cholesky

Considere o sistema linear

$$
Ax=b,
$$

em que

$$
A=
\begin{bmatrix}
6 & 2 & 1\\
2 & 5 & 2\\
1 & 2 & 4
\end{bmatrix},
\qquad
b=
\begin{bmatrix}
9\\
4\\
7
\end{bmatrix}.
$$

Resolva o sistema usando a decomposição de Cholesky. Organize sua solução nas etapas a seguir.

#### Parte A — Verificação das condições

1. Verifique se $A$ é simétrica.
2. Calcule os menores principais líderes de $A$ e confirme se ela é definida positiva.
3. Explique por que a matriz pode ser fatorada pelo método de Cholesky.

#### Parte B — Fatoração

4. Encontre uma matriz triangular inferior $L$ tal que

$$
A=LL^T.
$$

5. Confira a fatoração calculando o produto $LL^T$.

#### Parte C — Resolução do sistema

6. Resolva o sistema triangular

$$
Ly=b
$$

por substituição progressiva.

7. Resolva o sistema

$$
L^T x=y
$$

por substituição regressiva.

8. Apresente a solução aproximada de $Ax=b$.

#### Parte D — Verificação

9. Calcule o resíduo

$$
r=b-Ax.
$$

10. Verifique se o resíduo é próximo de zero e interprete o resultado.

**Entrega:** mostre os cálculos da fatoração, as duas substituições e a verificação final. Você pode usar Python para conferir os resultados, mas apresente também o desenvolvimento algébrico.

Cóigo opcional para conferir a resposta:

In [8]:
import numpy as np

A = np.array([
    [6.0, 2.0, 1.0],
    [2.0, 5.0, 2.0],
    [1.0, 2.0, 4.0]
])

b = np.array([9.0, 4.0, 7.0])

L = np.linalg.cholesky(A)
y = np.linalg.solve(L, b)
x = np.linalg.solve(L.T, y)
residuo = b - A @ x

print("L =\n", L)
print("y =", y)
print("x =", x)
print("Resíduo b - Ax =", residuo)

L =
 [[2.44949  0.       0.      ]
 [0.816497 2.081666 0.      ]
 [0.408248 0.800641 1.786703]]
y = [3.674235 0.480384 2.86303 ]
x = [ 1.361446 -0.385542  1.60241 ]
Resíduo b - Ax = [ 0.  0. -0.]


### Gabarito: decomposição de Cholesky

Considere

$$
A=
\begin{bmatrix}
6 & 2 & 1\\
2 & 5 & 2\\
1 & 2 & 4
\end{bmatrix},
\qquad
b=
\begin{bmatrix}
9\\
4\\
7
\end{bmatrix}.
$$

Queremos resolver $Ax=b$ usando a decomposição de Cholesky.

#### 1. Verificar se a matriz é simétrica definida positiva

A matriz é simétrica, pois $A=A^T$.

Seus menores principais líderes são:

$$
\Delta_1=6>0,
$$

$$
\Delta_2=
\begin{vmatrix}
6 & 2\\
2 & 5
\end{vmatrix}
=6\cdot5-2\cdot2
=26>0,
$$

e

$$
\Delta_3=\det(A)=81>0.
$$

Como $A$ é simétrica e todos os menores principais líderes são positivos, o critério de Sylvester garante que $A$ é definida positiva. Portanto, podemos aplicar Cholesky.

#### 2. Calcular a fatoração $A=LL^T$

Procuramos uma matriz triangular inferior

$$
L=
\begin{bmatrix}
l_{11} & 0 & 0\\
l_{21} & l_{22} & 0\\
l_{31} & l_{32} & l_{33}
\end{bmatrix}
$$

tal que $LL^T=A$.

Calculamos os elementos de $L$:

$$
l_{11}=\sqrt{6}.
$$

$$
l_{21}=\frac{2}{l_{11}}
=\frac{2}{\sqrt{6}}.
$$

$$
l_{31}=\frac{1}{l_{11}}
=\frac{1}{\sqrt{6}}.
$$

$$
l_{22}
=
\sqrt{5-l_{21}^2}
=
\sqrt{5-\frac{4}{6}}
=
\sqrt{\frac{13}{3}}.
$$

$$
l_{32}
=
\frac{2-l_{31}l_{21}}{l_{22}}
=
\frac{2-\frac{1}{3}}{\sqrt{13/3}}
=
\frac{5}{\sqrt{39}}.
$$

$$
l_{33}
=
\sqrt{4-l_{31}^2-l_{32}^2}
=
\sqrt{4-\frac{1}{6}-\frac{25}{39}}
=
\sqrt{\frac{34}{13}}.
$$

Logo,

$$
L=
\begin{bmatrix}
\sqrt{6} & 0 & 0\\
\frac{2}{\sqrt{6}} & \sqrt{\frac{13}{3}} & 0\\
\frac{1}{\sqrt{6}} & \frac{5}{\sqrt{39}} & \sqrt{\frac{34}{13}}
\end{bmatrix}.
$$

#### 3. Resolver $Ly=b$ por substituição progressiva

O primeiro sistema é

$$
\begin{bmatrix}
\sqrt{6} & 0 & 0\\
\frac{2}{\sqrt{6}} & \sqrt{\frac{13}{3}} & 0\\
\frac{1}{\sqrt{6}} & \frac{5}{\sqrt{39}} & \sqrt{\frac{34}{13}}
\end{bmatrix}
\begin{bmatrix}
y_1\\
y_2\\
y_3
\end{bmatrix}
=
\begin{bmatrix}
9\\
4\\
7
\end{bmatrix}.
$$

Da primeira linha:

$$
\sqrt{6}\,y_1=9
\quad\Longrightarrow\quad
y_1=\frac{9}{\sqrt{6}}.
$$

Da segunda linha:

$$
\frac{2}{\sqrt{6}}y_1+\sqrt{\frac{13}{3}}\,y_2=4.
$$

Substituindo $y_1=9/\sqrt{6}$:

$$
3+\sqrt{\frac{13}{3}}\,y_2=4
\quad\Longrightarrow\quad
y_2=\sqrt{\frac{3}{13}}.
$$

Da terceira linha:

$$
\frac{1}{\sqrt{6}}y_1+
\frac{5}{\sqrt{39}}y_2+
\sqrt{\frac{34}{13}}\,y_3=7.
$$

Substituindo os valores já calculados:

$$
\frac{3}{2}+\frac{5}{13}
+\sqrt{\frac{34}{13}}\,y_3=7,
$$

$$
\sqrt{\frac{34}{13}}\,y_3=\frac{109}{26}
\quad\Longrightarrow\quad
y_3=\frac{109}{26}\sqrt{\frac{13}{34}}.
$$

Portanto,

$$
y=
\begin{bmatrix}
\frac{9}{\sqrt{6}}\\
\sqrt{\frac{3}{13}}\\
\frac{109}{26}\sqrt{\frac{13}{34}}
\end{bmatrix}.
$$

#### 4. Resolver $L^T x=y$ por substituição regressiva

Agora resolvemos

$$
L^T x=y,
$$

com

$$
L^T=
\begin{bmatrix}
\sqrt{6} & \frac{2}{\sqrt{6}} & \frac{1}{\sqrt{6}}\\
0 & \sqrt{\frac{13}{3}} & \frac{5}{\sqrt{39}}\\
0 & 0 & \sqrt{\frac{34}{13}}
\end{bmatrix}.
$$

Da terceira linha:

$$
\sqrt{\frac{34}{13}}\,x_3=y_3
\quad\Longrightarrow\quad
x_3=\frac{109}{68}.
$$

Da segunda linha:

$$
\sqrt{\frac{13}{3}}\,x_2+
\frac{5}{\sqrt{39}}x_3=y_2,
$$

$$
x_2=-\frac{10}{17}.
$$

Da primeira linha:

$$
\sqrt{6}\,x_1+
\frac{2}{\sqrt{6}}x_2+
\frac{1}{\sqrt{6}}x_3=y_1,
$$

$$
x_1=\frac{47}{34}.
$$

Assim, a solução é

$$
x=
\begin{bmatrix}
\frac{47}{34}\\
-\frac{10}{17}\\
\frac{109}{68}
\end{bmatrix}.
$$

#### 5. Verificar a solução

Substituindo $x$ no sistema original:

$$
Ax=
\begin{bmatrix}
6 & 2 & 1\\
2 & 5 & 2\\
1 & 2 & 4
\end{bmatrix}
\begin{bmatrix}
\frac{47}{34}\\
-\frac{10}{17}\\
\frac{109}{68}
\end{bmatrix}
=
\begin{bmatrix}
9\\
4\\
7
\end{bmatrix}
=b.
$$

Logo, o resíduo é

$$
r=b-Ax=
\begin{bmatrix}
0\\
0\\
0
\end{bmatrix},
$$

e a solução está correta.